# 🌎 ClimaAnalytics — Big Data + Machine Learning with Apache Spark

> **Statistical analysis and K-Means clustering of 1.6M weather records on Databricks**

**Authors:** Andrés Felipe Hinojosa Galindo · Diego Alexander Millan Gualdron
**University:** UNICIENCIA — Systems Engineering
**Date:** May 2026
**Stack:** Apache Spark · PySpark ML · Spark SQL · Databricks · K-Means
**Dataset:** [Historical Hourly Weather Data — 36 Cities (Kaggle)](https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data)

---

## 📌 Executive summary

Can we automatically group 36 cities around the world into homogeneous climate profiles without manual labeling? And which variable — temperature, humidity, pressure, or wind — best differentiates these groups?

This project answers both questions by processing **1,629,108 hourly weather records** (5 years, 36 cities) with Apache Spark on Databricks, applying a PySpark ML Pipeline with K-Means (k=5) and evaluating it via Silhouette Score.

**Key finding:** **relative humidity**, not temperature, is the variable that most differentiates climates. A counterintuitive result with practical applications in agriculture, energy, tourism, and climate risk modeling.

---

## 🗂️ Table of contents
1. [Setup and data ingestion](#1-setup)
2. [Wide → Long transformation](#2-transformation)
3. [EDA and quality diagnostics](#3-eda)
4. [Cleaning and feature engineering](#4-cleaning)
5. [Spark SQL analytics](#5-sql)
6. [Pearson correlation](#6-correlation)
7. [K-Means model with ML Pipeline](#7-kmeans)
8. [Evaluation with Silhouette Score](#8-evaluation)
9. [Results and interpretation](#9-results)
10. [Conclusions and biomedical applications](#10-conclusions)

## 1. Setup and data ingestion <a id="1-setup"></a>

The original dataset consists of **7 CSV files** in wide format, where each column represents a city. Spark reads them in a distributed fashion from a Unity Catalog volume on Databricks.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

# Base path of the Unity Catalog volume on Databricks
BASE = "/Volumes/workspace/default/clima_g5"

# Distributed read of the main CSVs
temp_df     = spark.read.csv(f"{BASE}/temperature.csv",         header=True, inferSchema=True)
humidity_df = spark.read.csv(f"{BASE}/humidity.csv",            header=True, inferSchema=True)
pressure_df = spark.read.csv(f"{BASE}/pressure.csv",            header=True, inferSchema=True)
wind_spd_df = spark.read.csv(f"{BASE}/wind_speed.csv",          header=True, inferSchema=True)
wind_dir_df = spark.read.csv(f"{BASE}/wind_direction.csv",      header=True, inferSchema=True)

print(f"✅ Files loaded — columns in temperature.csv: {len(temp_df.columns)}")

## 2. Wide → Long transformation <a id="2-transformation"></a>

**The most interesting technical challenge of the project.** The CSVs come in wide format (1 column per city), but for clustering we need long format (1 row per `(city, timestamp)` pair).

The technique uses PySpark's `explode()` + `create_map()` to do this **in a distributed way** without crashing the cluster:

`45,253 timestamps × 36 cities = 1,629,108 records`

In [ ]:
def wide_to_long(df, value_col_name):
    """Transforms a wide-format DataFrame (one column per city) into long format."""
    city_cols = [c for c in df.columns if c != "datetime"]
    return df.select(
        F.col("datetime"),
        F.explode(F.create_map(*[
            item for city in city_cols
            for item in (F.lit(city), F.col(f"`{city}`").cast(DoubleType()))
        ])).alias("city", value_col_name)
    )

# Apply to each variable
temp_long = wide_to_long(temp_df,     "temperature")
hum_long  = wide_to_long(humidity_df, "humidity")
pres_long = wide_to_long(pressure_df, "pressure")
wspd_long = wide_to_long(wind_spd_df, "wind_speed")
wdir_long = wide_to_long(wind_dir_df, "wind_direction")

# Join all variables into a single master DataFrame
df = (temp_long
      .join(hum_long,  ["datetime", "city"])
      .join(pres_long, ["datetime", "city"])
      .join(wspd_long, ["datetime", "city"])
      .join(wdir_long, ["datetime", "city"]))

print(f"✅ Total records after transformation: {df.count():,}")

## 3. EDA and quality diagnostics <a id="3-eda"></a>

Before modeling, we validate that the data is properly typed and that there are no critical nulls. The diagnosis is performed with a single pass over the DataFrame to avoid wasting cluster resources.

In [ ]:
# Schema and descriptive statistics
df.printSchema()
df.describe().show()

# Null diagnosis in a single pass (important for large datasets)
total = df.count()
nulls_df = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
    for c in df.columns
]).toPandas().T.reset_index()
nulls_df.columns = ["column", "nulls"]
nulls_df["percentage"] = (nulls_df["nulls"] / total * 100).round(2).astype(str) + "%"
print(f"\nNull diagnosis over {total:,} records:")
print(nulls_df.sort_values("nulls", ascending=False).to_string(index=False))

## 4. Cleaning and feature engineering <a id="4-cleaning"></a>

Two derived columns that add analytical value:
- **`temp_celsius`** — the original dataset comes in Kelvin
- **`wind_category`** — simplified Beaufort categorization of wind speed

In [ ]:
# Drop duplicates
df = df.dropDuplicates()

# Derived column 1: temperature in Celsius
df = df.withColumn("temp_celsius", F.col("temperature") - 273.15)

# Derived column 2: wind category (simplified Beaufort scale)
df = df.withColumn("wind_category",
    F.when(F.col("wind_speed") < 3,  "Calm")
     .when(F.col("wind_speed") < 8,  "Light breeze")
     .when(F.col("wind_speed") < 14, "Moderate breeze")
     .otherwise("Strong wind")
)

df.select("city", "temperature", "temp_celsius", "wind_speed", "wind_category").show(5)

## 5. Spark SQL analytics <a id="5-sql"></a>

Spark SQL allows writing declarative queries on distributed DataFrames. Three representative queries:

In [ ]:
# Register temporary view for Spark SQL
df_clean = df.withColumn("datetime_ts", F.to_timestamp(F.col("datetime")))
df_clean.createOrReplaceTempView("weather")

# Query 1: Top 10 hottest cities
print("=== TOP 10 hottest cities ===")
spark.sql("""
    SELECT city,
           ROUND(AVG(temp_celsius), 2) AS avg_temp_c,
           ROUND(AVG(humidity), 2)     AS avg_humidity,
           COUNT(*)                    AS total_records
    FROM weather
    GROUP BY city
    ORDER BY avg_temp_c DESC
    LIMIT 10
""").show()

# Query 2: Yearly temperature trend
print("=== Yearly temperature trend ===")
spark.sql("""
    SELECT YEAR(datetime_ts) AS year,
           ROUND(AVG(temp_celsius), 2) AS avg_temp_c
    FROM weather
    WHERE datetime_ts IS NOT NULL
    GROUP BY year
    ORDER BY year
""").show()

# Query 3: Cities with humidity > 70% (HAVING)
print("=== Cities with average humidity > 70% ===")
spark.sql("""
    SELECT city, ROUND(AVG(humidity), 2) AS avg_humidity
    FROM weather
    GROUP BY city
    HAVING avg_humidity > 70
    ORDER BY avg_humidity DESC
""").show()

## 6. Pearson correlation <a id="6-correlation"></a>

A first check of linear relationships between variables. Spoiler: correlations are weak, hinting that climate patterns are **non-linear** — which justifies the use of a clustering model.

In [ ]:
corr_temp_hum  = df.stat.corr("temperature", "humidity")
corr_temp_wind = df.stat.corr("temperature", "wind_speed")

print(f"Correlation Temperature vs Humidity:   {corr_temp_hum:.4f}")
print(f"Correlation Temperature vs Wind speed: {corr_temp_wind:.4f}")

# Interpretation according to standard Pearson thresholds
for name, value in [("Temp-Humidity", corr_temp_hum), ("Temp-Wind", corr_temp_wind)]:
    if abs(value) > 0.5:   level = "STRONG"
    elif abs(value) > 0.3: level = "MODERATE"
    else:                  level = "WEAK"
    print(f"➡ {name}: {level} correlation")

## 7. K-Means model with ML Pipeline <a id="7-kmeans"></a>

A 3-stage pipeline following PySpark ML best practices:

| Stage | Function | Why it matters |
|-------|----------|----------------|
| 1. `VectorAssembler` | Combines features into a vector | Required by the PySpark ML API |
| 2. `StandardScaler` | Normalizes by standard deviation | **Critical** — without it, pressure (~1000 hPa) would dominate over wind (~2 m/s) |
| 3. `KMeans` (k=5) | Assigns each record to a cluster | k=5 was the project requirement |

In [ ]:
# Climate variables for clustering
feature_cols = ["temperature", "humidity", "pressure", "wind_speed", "wind_direction"]

# Full ML Pipeline
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features")
kmeans    = KMeans(featuresCol="features", predictionCol="cluster", k=5, seed=42)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])

print("⏳ Training the model on 1.6M records...")
model = pipeline.fit(df)
df_pred = model.transform(df)
print("✅ Model trained")

df_pred.select("city", "temp_celsius", "humidity", "cluster").show(5)

## 8. Evaluation with Silhouette Score <a id="8-evaluation"></a>

The **Silhouette Score** measures how well-separated the clusters are. It ranges from -1 to 1, where values close to 1 indicate compact, well-separated groups.

> ⚠️ **Critical interpretation note:** a "low" score on hourly weather data is **not a failure**. The same city changes its climate profile across seasons — Chicago in July and Chicago in January are different climates. Cluster overlap is expected and reflects the reality of the modeled phenomenon.

In [ ]:
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette"
)

score = evaluator.evaluate(df_pred)
print(f"✅ Silhouette Score: {score:.4f}")

if score > 0.5:   print("➡ GOOD-quality clustering")
elif score > 0.3: print("➡ ACCEPTABLE quality")
else:             print("➡ Low score — expected on hourly data with seasonal variation")

## 9. Results and interpretation <a id="9-results"></a>

### 🌍 The 5 climate profiles identified

| Cluster | Avg. temp | Humidity | Records | Climate profile |
|---------|-----------|----------|---------|-----------------|
| 0 | 12.83°C | 76.91% | 467,429 | **Cold and humid** |
| 1 | 18.22°C | 80.27% | 481,351 | **Mild and very humid** |
| 2 | 24.54°C | 40.56% | 377,142 | **Hot and dry** |
| 3 | 3.55°C  | 70.09% | 296,894 | **Very cold and windy** |
| 4 | 14.61°C | 49.61% | 6,292   | Atypical (low pressure) |

### 🏙️ Dominant cluster per city

- **Eilat (Israel)** → 55.6% in Cluster 2 (Hot and dry) ✅ Consistent with its desert geography
- **Chicago, Boston, Detroit** → Cluster 3 (Very cold and windy) ✅ Consistent with the U.S. Northeast
- **Atlanta** → Cluster 1 (Mild and humid) ✅ Consistent with Southeast U.S. climate

### 🎯 Key finding: humidity matters more than temperature

In [ ]:
# Centroid analysis to identify the most discriminative variable
import numpy as np

kmeans_model = model.stages[-1]

print("Normalized centroids of each cluster:\n")
for i, centroid in enumerate(kmeans_model.clusterCenters()):
    print(f"Cluster {i}:")
    for col_name, val in zip(feature_cols, centroid):
        print(f"  {col_name:15s}: {val:.4f}")
    print()

# Compute the range of each feature across clusters
centroids = np.array(kmeans_model.clusterCenters())
print("Range of each variable across clusters (larger range = stronger discriminative power):")
for i, col_name in enumerate(feature_cols):
    rng = centroids[:, i].max() - centroids[:, i].min()
    print(f"  {col_name:15s}: range = {rng:.2f}")

**Centroid analysis result:**

| Variable | Range across clusters | Discriminative power |
|----------|----------------------|----------------------|
| **Humidity** | 1.81 – 3.59 | 🥇 **VERY HIGH** |
| Temperature | 26.80 – 28.83 | 🥈 HIGH |
| Wind speed | 0.86 – 2.28 | MEDIUM |
| Wind direction | 0.68 – 2.37 | MEDIUM |
| Pressure | 54.83 – 65.89 | LOW |

> 💡 **Analytical conclusion:** relative humidity, not temperature, defines whether a city belongs to a dry or humid climate profile. This result directly answers the project's business question.

## 10. Conclusions and biomedical applications <a id="10-conclusions"></a>

### ✅ What the project demonstrated technically
1. **Distributed processing justified:** 1,629,108 records far exceed the threshold where pandas becomes inefficient.
2. **Wide→long transformation with `explode()` + `create_map()`** is the correct pattern for multivariate datasets in Spark.
3. **Reproducible and scalable pipeline:** the Ingestion → EDA → Cleaning → SQL → ML → Evaluation flow is reusable on 10M+ records without code changes.
4. **Actionable finding:** relative humidity is the variable that most differentiates climates, not temperature.

### 🌐 Multi-sector applications of this pipeline
- **Agriculture:** replicate successful crops between cities in the same cluster.
- **Energy:** forecast heating demand in cold clusters.
- **Tourism:** classify destinations by climate profile.
- **Insurance:** model regional climate risk.

---

### 🩺 Application to the biomedical domain (my focus area)

The techniques used in this project are **identical** to those applied in clinical and biomedical research:

| Here (climate) | In biomedical research |
|----------------|------------------------|
| Clustering cities by climate variables | **Patient phenotyping** by physiological variables |
| K-Means on 1.6M hourly records | Clustering on time-series data from wearables, EEG, ECG |
| Silhouette Score to validate groups | Same score to validate disease subtypes |
| Distributed pipeline on Databricks | Genomic or EHR data processing at scale |
| Centroids → most discriminative variable | Identification of key biomarkers in clinical clusters |

> **My professional value proposition:** depth in biomedical data, versatility to solve problems with data from any domain.

---

## 🔗 References
- Dataset: [Historical Hourly Weather Data — Kaggle](https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data)
- [Apache Spark MLlib documentation](https://spark.apache.org/docs/latest/ml-guide.html)
- [Databricks Free Edition docs](https://docs.databricks.com/aws/en/getting-started/free-edition)
- Zaharia, M. et al. (2010). Spark: Cluster computing with working sets. *HotCloud*.

---

**📫 Contact:** [LinkedIn] · [GitHub]